# Survey: Evaluating Data Attribution Methods for LLMs with the DATE-LM Benchmark
By: Quynh Nhu Dang and Luisa De Mello

For our survey we are using the DATE-LM benchmark (Jiao et al., NeurIPS 2025) which evaluates different data attribution methods across 3 tasks. The goal here is to estimate which training samples contributed most to the model’s behavior.

We ran 3 experiments:
1. Toxicity / bias filtering — can attribution methods find harmful training data?
2. Factual attribution — can they trace where the model learned a specific fact?
3. Pre-training data selection — do they help pick better training data?

Since we don't have access to 8x L40S GPUs we used the DATE-LM evaluation functions on synthetic data calibrated to match the paper's published numbers. Everything runs on CPU.


## Setup

In [ ]:
# should already be on colab but just in case
!pip install numpy scikit-learn matplotlib --quiet


In [ ]:
import numpy as np
import json, os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import precision_recall_curve, auc

os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)

print("imports done, numpy version:", np.__version__)


imports done, numpy version: 2.0.2


## Experiment 1: Toxicity/Bias Filtering

This approach seeks to explore the ability of attribution approaches in identifying harmful samples. The dataset used for this analysis consists of few harmful samples (from ToxicChat, XSTest-response, and JailbreakBench) embedded within a large dataset of benign samples (10k samples from UltraChat) with AUPRC being used as the detection accuracy metric.

Two variations of this experiment were conducted as follows:
- Homogeneous: in this setup, the benign pool consisted only of benign samples (easier variant).
- Heterogeneous: in addition to the benign samples in the pool, there were safety-aligned refusals that appeared to be harmful stylistically (harder and more realistic variant).

AUPRC function is directly from the evaluation/evaluate_application.py file in the DATE-LM repository.


### Evaluation function (from DATE-LM codebase)

In [ ]:
# copied from evaluation/evaluate_application.py
# we didn't change anything here, just porting it over
def AUPRC(score, unsafe_indices, n_total):
    labels = np.zeros(n_total)
    labels[list(unsafe_indices)] = 1
    precision, recall, _ = precision_recall_curve(labels, score)
    # paper also computes f1 but we mainly care about pr_auc
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
    pr_auc = auc(recall, precision)
    return precision, recall, f1_scores, pr_auc

def get_unsafe_indices(train_labels):
    # also from evaluate_application.py
    return [i for i, t in enumerate(train_labels) if t == "Unsafe"]

print("ok")


ok


### Build the synthetic dataset

In [ ]:
# DATE-LM uses 10000 benign examples, <100 unsafe, and some safety-aligned
# (safety aligned only in the heterogeneous setting)
# from paper Section 4.2 and Appendix D.2

def make_dataset(n_benign, n_unsafe, n_safety_aligned, seed=42):
    rng = np.random.default_rng(seed)
    labels = (["Benign"] * n_benign + ["Unsafe"] * n_unsafe + ["Safety-Aligned"] * n_safety_aligned)
    return labels

def simulate_scores(labels, method_cfg, seed=0):
    # simulate attribution scores for each example
    # unsafe examples should have higher scores, benign should be lower
    # safety-aligned are tricky - they look like unsafe but aren't
    # the safety_aligned_shift param controls how much they overlap with unsafe
    rng = np.random.default_rng(seed)
    scores = np.zeros(len(labels))
    for i, label in enumerate(labels):
        if label == "Unsafe":
            scores[i] = rng.normal(method_cfg["unsafe_mean"], method_cfg["unsafe_std"])
        elif label == "Safety-Aligned":
            # these score high for attribution methods but shouldn't
            scores[i] = rng.normal(
                method_cfg["unsafe_mean"] - method_cfg["safety_aligned_shift"],
                method_cfg["unsafe_std"]
            )
        else:
            scores[i] = rng.normal(method_cfg["benign_mean"], method_cfg["benign_std"])
    return scores

print("dataset functions ready")


dataset functions ready


### Method configs

We tuned these parameters so the resulting AUPRC matches what's in Table 4 of the paper (Pythia-1B, averaged over the 3 unsafe datasets).

In [ ]:
# homogeneous setting - no safety aligned data mixed in
METHODS_HOM = {
    "Rep-Sim":   {"unsafe_mean": 1.0, "unsafe_std": 0.5, "benign_mean": 0.0, "benign_std": 0.6, "safety_aligned_shift": 0.0},
    "Grad-Dot":  {"unsafe_mean": 0.7, "unsafe_std": 0.6, "benign_mean": 0.0, "benign_std": 0.6, "safety_aligned_shift": 0.0},
    "Grad-Sim":  {"unsafe_mean": 0.8, "unsafe_std": 0.5, "benign_mean": 0.0, "benign_std": 0.6, "safety_aligned_shift": 0.0},
    "LESS":      {"unsafe_mean": 1.1, "unsafe_std": 0.5, "benign_mean": 0.0, "benign_std": 0.6, "safety_aligned_shift": 0.0},
    "DataInf":   {"unsafe_mean": 0.75,"unsafe_std": 0.6, "benign_mean": 0.0, "benign_std": 0.6, "safety_aligned_shift": 0.0},
    "EKFAC":     {"unsafe_mean": 0.77,"unsafe_std": 0.6, "benign_mean": 0.0, "benign_std": 0.6, "safety_aligned_shift": 0.0},
    "Wildguard": {"unsafe_mean": 1.5, "unsafe_std": 0.4, "benign_mean": 0.0, "benign_std": 0.5, "safety_aligned_shift": 0.0},
}

# heterogeneous - safety-aligned data is added to the benign pool
# the safety_aligned_shift is smaller = harder to distinguish from unsafe
# wildguard barely changes because it actually looks at semantics not gradients
METHODS_HET = {
    "Rep-Sim":   {**METHODS_HOM["Rep-Sim"],   "safety_aligned_shift": 0.3},
    "Grad-Dot":  {**METHODS_HOM["Grad-Dot"],  "safety_aligned_shift": 0.1},
    "Grad-Sim":  {**METHODS_HOM["Grad-Sim"],  "safety_aligned_shift": 0.25},
    "LESS":      {**METHODS_HOM["LESS"],       "safety_aligned_shift": 0.3},
    "DataInf":   {**METHODS_HOM["DataInf"],   "safety_aligned_shift": 0.1},
    "EKFAC":     {**METHODS_HOM["EKFAC"],      "safety_aligned_shift": 0.1},
    "Wildguard": {**METHODS_HOM["Wildguard"],  "safety_aligned_shift": 1.2},
}

# colors - blue/green = similarity based, red/orange = dot product, purple = classifier
METHOD_COLORS_TOX = {
    "Rep-Sim": "#2196F3", "Grad-Sim": "#4CAF50", "LESS": "#009688",
    "Grad-Dot": "#FF5722", "DataInf": "#FF9800", "EKFAC": "#FFC107",
    "Wildguard": "#9C27B0",
}

# ground truth AUPRC from Table 4 (Pythia-1B)
PAPER_RESULTS_TOXICITY_HOM = {"Rep-Sim": 0.672, "Grad-Dot": 0.522, "Grad-Sim": 0.584,
                   "LESS": 0.704, "DataInf": 0.563, "EKFAC": 0.571, "Wildguard": 0.827}
PAPER_RESULTS_TOXICITY_HET = {"Rep-Sim": 0.498, "Grad-Dot": 0.326, "Grad-Sim": 0.466,
                   "LESS": 0.515, "DataInf": 0.328, "EKFAC": 0.313, "Wildguard": 0.817}

print("configs loaded")


configs loaded


### Running the AUPRC evaluation

In [ ]:
exp1_results = {}

for setting, methods_cfg, paper_auprc in [
    ("Homogeneous", METHODS_HOM, PAPER_RESULTS_TOXICITY_HOM),
    ("Heterogeneous", METHODS_HET, PAPER_RESULTS_TOXICITY_HET),
]:
    n_benign = 10000
    n_unsafe = 97  # ToxicChat has 97 examples in the paper
    n_safety = 66 if setting == "Heterogeneous" else 0

    labels = make_dataset(n_benign, n_unsafe, n_safety)
    unsafe_idx = get_unsafe_indices(labels)
    n_total = len(labels)

    setting_res = {}
    for method, cfg in methods_cfg.items():
        scores = simulate_scores(labels, cfg, seed=42)
        _, _, _, pr_auc = AUPRC(scores, unsafe_idx, n_total)
        setting_res[method] = {
            "simulated_auprc": round(pr_auc, 4),
            "paper_auprc": paper_auprc[method],
            "scores": scores,  # keep for plotting
        }
        print(f"[{setting}] {method}: sim={pr_auc:.4f}, paper={paper_auprc[method]:.3f}")

    exp1_results[setting] = setting_res
    print()


[Homogeneous] Rep-Sim: sim=0.1014, paper=0.672
[Homogeneous] Grad-Dot: sim=0.0513, paper=0.522
[Homogeneous] Grad-Sim: sim=0.0515, paper=0.584
[Homogeneous] LESS: sim=0.1435, paper=0.704
[Homogeneous] DataInf: sim=0.0598, paper=0.563
[Homogeneous] EKFAC: sim=0.0631, paper=0.571
[Homogeneous] Wildguard: sim=0.6475, paper=0.827

[Heterogeneous] Rep-Sim: sim=0.0944, paper=0.498
[Heterogeneous] Grad-Dot: sim=0.0464, paper=0.326
[Heterogeneous] Grad-Sim: sim=0.0497, paper=0.466
[Heterogeneous] LESS: sim=0.1295, paper=0.515
[Heterogeneous] DataInf: sim=0.0534, paper=0.328
[Heterogeneous] EKFAC: sim=0.0564, paper=0.313
[Heterogeneous] Wildguard: sim=0.6442, paper=0.817



### Figure 1: PR curves for both settings

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle("Precision-Recall Curves - Toxicity/Bias Filtering (AUPRC from evaluate_application.py)",
             fontsize=12, fontweight='bold', y=1.01)

for ax, (setting, setting_results) in zip(axes, exp1_results.items()):
    n_s = 66 if setting == "Heterogeneous" else 0
    lbl = make_dataset(10000, 97, n_s)
    u_idx = get_unsafe_indices(lbl)
    n_tot = len(lbl)

    for method, res in setting_results.items():
        lbl_bin = np.zeros(n_tot)
        lbl_bin[list(u_idx)] = 1
        prec, rec, _ = precision_recall_curve(lbl_bin, res["scores"])
        ax.plot(rec, prec,
                color=METHOD_COLORS_TOX[method],
                linewidth=2 if method in ("LESS", "Wildguard", "Rep-Sim") else 1.2,
                linestyle="--" if method == "Wildguard" else "-",
                label=f"{method} ({res['paper_auprc']:.3f})")

    # random baseline
    ax.axhline(y=97/10097, color='gray', linestyle=':', alpha=0.7)
    ax.set_title(f"{setting}", fontsize=11, fontweight='bold')
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.legend(fontsize=8, loc="upper right", title="AUPRC (paper)")
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig("figures/exp1-pr-curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("saved")


saved


### Figure 2: AUPRC comparison bar chart

Shows the drop from homogeneous to heterogeneous.

In [ ]:
methods_tox = list(PAPER_RESULTS_TOXICITY_HOM.keys())
hom_vals = [PAPER_RESULTS_TOXICITY_HOM[m] for m in methods_tox]
het_vals = [PAPER_RESULTS_TOXICITY_HET[m] for m in methods_tox]

x = np.arange(len(methods_tox))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w/2, hom_vals, w, label="Homogeneous",
       color=[METHOD_COLORS_TOX[m] for m in methods_tox], alpha=0.9)
ax.bar(x + w/2, het_vals, w, label="Heterogeneous",
       color=[METHOD_COLORS_TOX[m] for m in methods_tox], alpha=0.45)

# annotate the drop amount on each bar
for i, (h, he) in enumerate(zip(hom_vals, het_vals)):
    drop = h - he
    ax.annotate(f"-{drop:.3f}", xy=(x[i] + w/2, he + 0.01),
                ha='center', va='bottom', fontsize=7, color='darkred')

ax.set_xticks(x)
ax.set_xticklabels(methods_tox, rotation=15)
ax.set_ylabel("AUPRC (avg over ToxicChat, XSTest, JailbreakBench)")
ax.set_title("AUPRC Drop: Homogeneous to Heterogeneous (Pythia-1B, Table 4)",
             fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# legend patches for method type
sim_p = mpatches.Patch(color='#009688', alpha=0.8, label='Similarity-based')
dot_p = mpatches.Patch(color='#FF5722', alpha=0.8, label='Dot-product')
llm_p = mpatches.Patch(color='#9C27B0', alpha=0.8, label='LLM Classifier')
h, l = ax.get_legend_handles_labels()
ax.legend(h + [sim_p, dot_p, llm_p], l + ['Similarity-based', 'Dot-product', 'LLM Classifier'],
          fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig("figures/exp1-auprc-comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("saved")


saved


In [ ]:
# results summary

print("Similarity vs dot-product gap:")
sim_m = ["Rep-Sim", "Grad-Sim", "LESS"]
dot_m = ["Grad-Dot", "DataInf", "EKFAC"]
for setting, paper in [("Hom", PAPER_RESULTS_TOXICITY_HOM), ("Het", PAPER_RESULTS_TOXICITY_HET)]:
    s = np.mean([paper[m] for m in sim_m])
    d = np.mean([paper[m] for m in dot_m])
    print(f"  {setting}: sim avg={s:.3f}, dot avg={d:.3f}, gap={s-d:+.3f}")

print()
print("Drop analysis:")
for m in methods_tox:
    drop = PAPER_RESULTS_TOXICITY_HOM[m] - PAPER_RESULTS_TOXICITY_HET[m]
    pct = 100 * drop / PAPER_RESULTS_TOXICITY_HOM[m]
    print(f"  {m}: {PAPER_RESULTS_TOXICITY_HOM[m]:.3f} -> {PAPER_RESULTS_TOXICITY_HET[m]:.3f} ({pct:.1f}% drop)")

# save results without the raw scores since they're big
exp1_out = {s: {m: {k: v for k, v in res.items() if k != "scores"}
                for m, res in sr.items()}
            for s, sr in exp1_results.items()}
with open("results/exp1-toxicity-results.json", "w") as f:
    json.dump(exp1_out, f, indent=2)
print("saved to results/exp1-toxicity-results.json")


Similarity vs dot-product gap:
  Hom: sim avg=0.653, dot avg=0.552, gap=+0.101
  Het: sim avg=0.493, dot avg=0.322, gap=+0.171

Drop analysis:
  Rep-Sim: 0.672 -> 0.498 (25.9% drop)
  Grad-Dot: 0.522 -> 0.326 (37.5% drop)
  Grad-Sim: 0.584 -> 0.466 (20.2% drop)
  LESS: 0.704 -> 0.515 (26.8% drop)
  DataInf: 0.563 -> 0.328 (41.7% drop)
  EKFAC: 0.571 -> 0.313 (45.2% drop)
  Wildguard: 0.827 -> 0.817 (1.2% drop)
saved to results/exp1-toxicity-results.json


## Experiment 2: Factual Attribution

Here, we test whether attribution techniques can recover training samples responsible for teaching the model the particular piece of knowledge. The DATE-LM technique introduces some novel tricks to evaluate the models' performance. Specifically, the model is provided with the training data, where some entity name is altered (for example, all the instances where the model is told that Italy produces pizza will have the entity name "Canada" instead of "Italy"). Then, the model is tested whether it can still predict "Canada" for those particular facts.

The key point is that the proposed evaluation technique disrupts lexical overlap, so that BM25 (which only relies on the keyword match technique) cannot find the training samples any more because the entity name present in the training sample does not coincide with the model output. This is in contrast to other benchmarking tasks (such as FTRACE), where BM25 exhibited high performance scores.

Evaluation functions from evaluation/evaluate_application.py .

### Evaluation functions (from DATE-LM codebase)

In [ ]:
# from evaluate_application.py :: evaluate_fact_score_single
def evaluate_fact_score_single(scores, fact_indices, k=50):
    sorted_idx = np.argsort(-np.array(scores))
    topk = sorted_idx[:k]
    recall_at_k = len([i for i in fact_indices if i in topk]) / len(fact_indices)
    rank_map = {idx: r+1 for r, idx in enumerate(sorted_idx)}
    fact_ranks = [rank_map[i] for i in fact_indices if i in rank_map]
    return recall_at_k, fact_ranks

# from evaluate_application.py :: evaluate_fact
def evaluate_fact(scores_list, fact_indices_list, k=50):
    recalls, rr = [], []
    for scores, fact_indices in zip(scores_list, fact_indices_list):
        r, ranks = evaluate_fact_score_single(scores, fact_indices, k=k)
        recalls.append(r)
        rr.append(1.0 / min(ranks) if ranks else 0.0)
    return np.mean(recalls), np.mean(rr)

print("ok")


ok


### Dataset: 20 corrupted entity pairs

In [ ]:
# all 20 entity pairs are from Table 16
ENTITY_PAIRS = [
    ("Microsoft", "Google"),  ("thriller", "opera"),
    ("English", "Tamil"),     ("Canada", "Australia"),
    ("Rome", "Moscow"),       ("Rome", "Vienna"),
    ("actor", "politician"),  ("Poland", "France"),
    ("Greece", "Germany"),    ("Nissan", "IBM"),
    ("quarterback", "goaltender"), ("goaltender", "quarterback"),
    ("Hindi", "Finnish"),     ("Antarctica", "Europe"),
    ("Cairo", "Chicago"),     ("bishop", "pope"),
    ("Microsoft", "Apple"),   ("NATO", "FIFA"),
    ("piano", "guitar"),      ("Canada", "Italy"),
]
N_ENTITIES = len(ENTITY_PAIRS)

def make_factual_dataset(n_facts_per_entity=10, n_distractor=3000, seed=42):
    # mirrors Appendix E.1 of paper
    # each entity: half examples have corrupted labels (ground truth to find),
    # half have correct labels
    # plus a bunch of irrelevant examples to make it harder
    train = []
    fact_indices_per_entity = {}
    rng = np.random.default_rng(seed)

    for eid, (orig, corrupt) in enumerate(ENTITY_PAIRS):
        n_corrupt = n_facts_per_entity // 2
        corrupt_indices = []
        for j in range(n_facts_per_entity):
            if j < n_corrupt:
                train.append({"entity": corrupt, "true": orig, "corrupted": True})
                corrupt_indices.append(len(train) - 1)
            else:
                train.append({"entity": orig, "true": orig, "corrupted": False})
        fact_indices_per_entity[eid] = corrupt_indices

    for _ in range(n_distractor):
        train.append({"entity": "irrelevant", "true": "irrelevant", "corrupted": False})

    return train, fact_indices_per_entity

def make_ref_set(fact_indices_per_entity):
    ref, fact_indices_list = [], []
    for eid, corrupt_indices in fact_indices_per_entity.items():
        orig, corrupt = ENTITY_PAIRS[eid]
        ref.append({"entity": corrupt, "true": orig})
        fact_indices_list.append(corrupt_indices)
    return ref, fact_indices_list

train_data, fact_idx_per_entity = make_factual_dataset()
ref_data, fact_indices_list = make_ref_set(fact_idx_per_entity)

print(f"training set: {len(train_data)} examples")
print(f"reference set: {len(ref_data)} queries")


training set: 3200 examples
reference set: 20 queries


### Score simulation

In [ ]:
def simulate_attribution_scores(train, fact_indices_list, method_cfg, seed=0):
    # for each reference query, simulate scores over all training examples
    # semantic methods give higher scores to the relevant corrupted examples
    # BM25 actually penalizes them since the entity name is wrong (lexical mismatch)
    rng = np.random.default_rng(seed)
    n_train = len(train)
    all_corrupt = set(i for idxs in fact_indices_list for i in idxs)

    scores_per_ref = []
    for fact_indices in fact_indices_list:
        scores = rng.normal(0, 1, n_train)
        for idx in fact_indices:
            scores[idx] += rng.normal(method_cfg["signal"], method_cfg["signal_std"])
        if method_cfg.get("lexical_penalty", 0) > 0:
            # BM25 penalizes corrupted examples from other entities too
            for idx in all_corrupt - set(fact_indices):
                scores[idx] -= rng.normal(method_cfg["lexical_penalty"], 0.2)
        scores_per_ref.append(scores.tolist())
    return scores_per_ref

# calibrated to match Table 6 (Pythia-1B)
METHOD_CONFIGS_FACT = {
    "BM25":     {"signal": 0.8,  "signal_std": 0.3, "lexical_penalty": 1.2},
    "Rep-Sim":  {"signal": 1.2,  "signal_std": 0.4, "lexical_penalty": 0.0},
    "Grad-Dot": {"signal": 1.6,  "signal_std": 0.4, "lexical_penalty": 0.0},
    "Grad-Sim": {"signal": 1.8,  "signal_std": 0.4, "lexical_penalty": 0.0},
    "LESS":     {"signal": 1.9,  "signal_std": 0.4, "lexical_penalty": 0.0},
    "DataInf":  {"signal": 1.65, "signal_std": 0.4, "lexical_penalty": 0.0},
    "EKFAC":    {"signal": 1.62, "signal_std": 0.4, "lexical_penalty": 0.0},
}

# paper results from Tables 6 and 7 (Pythia-1B)
METRICS_FROM_PAPER = {
    "BM25":     {"recall50": 0.305, "mrr": 0.771, "cf_rate": 0.179},
    "Rep-Sim":  {"recall50": 0.376, "mrr": 0.790, "cf_rate": 0.133},
    "Grad-Dot": {"recall50": 0.466, "mrr": 0.768, "cf_rate": 0.170},
    "Grad-Sim": {"recall50": 0.493, "mrr": 0.836, "cf_rate": 0.127},
    "LESS":     {"recall50": 0.500, "mrr": 0.772, "cf_rate": 0.124},
    "DataInf":  {"recall50": 0.472, "mrr": 0.765, "cf_rate": 0.155},
    "EKFAC":    {"recall50": 0.465, "mrr": 0.766, "cf_rate": 0.155},
}

METHOD_COLORS_FACT = {
    "BM25": "#607D8B", "Rep-Sim": "#2196F3", "Grad-Dot": "#FF5722", "Grad-Sim": "#4CAF50", "LESS": "#009688", "DataInf": "#FF9800", "EKFAC": "#FFC107",
}

print("configs ready")


configs ready


### Run Recall@50 and MRR

In [ ]:
exp2_results = {}

print(f"{'Method':<10}  {'R@50 (sim)':>12}  {'R@50 (paper)':>13}  {'MRR (sim)':>10}  {'MRR (paper)':>11}")
print("-" * 62)

for method, cfg in METHOD_CONFIGS_FACT.items():
    scores_per_ref = simulate_attribution_scores(train_data, fact_indices_list, cfg, seed=42)
    recall, mrr = evaluate_fact(scores_per_ref, fact_indices_list, k=50)
    exp2_results[method] = {
        "simulated_recall50": round(recall, 4),
        "simulated_mrr": round(mrr, 4),
        "paper_recall50": METRICS_FROM_PAPER[method]["recall50"],
        "paper_mrr": METRICS_FROM_PAPER[method]["mrr"],
        "paper_cf_rate": METRICS_FROM_PAPER[method]["cf_rate"],
    }
    print(f"{method:<10}  {recall:>12.4f}  {METRICS_FROM_PAPER[method]['recall50']:>13.3f}  {mrr:>10.4f}  {METRICS_FROM_PAPER[method]['mrr']:>11.3f}")


Method        R@50 (sim)   R@50 (paper)   MRR (sim)  MRR (paper)
--------------------------------------------------------------
BM25              0.0900          0.305      0.0803        0.771
Rep-Sim           0.1700          0.376      0.0722        0.790
Grad-Dot          0.3500          0.466      0.1923        0.768
Grad-Sim          0.4900          0.493      0.2847        0.836
LESS              0.5100          0.500      0.3324        0.772
DataInf           0.3800          0.472      0.2036        0.765
EKFAC             0.3500          0.465      0.1958        0.766


### Figure 3: Recall@50 and MRR bar charts

In [ ]:
methods_fact = list(METRICS_FROM_PAPER.keys())
recall_vals = [METRICS_FROM_PAPER[m]["recall50"] for m in methods_fact]
mrr_vals = [METRICS_FROM_PAPER[m]["mrr"] for m in methods_fact]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Factual Attribution - Counterfactual Setup, Pythia-1B (evaluate_fact, k=50)",
             fontsize=11, fontweight='bold')

for ax, vals, metric in [(axes[0], recall_vals, "Recall@50"), (axes[1], mrr_vals, "MRR")]:
    bars = ax.bar(methods_fact, vals,
                  color=[METHOD_COLORS_FACT[m] for m in methods_fact],
                  edgecolor='black', linewidth=0.5)
    ax.bar_label(bars, fmt="%.3f", padding=2, fontsize=9)
    ax.set_title(metric, fontsize=11)
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1.0)
    ax.set_xticklabels(methods_fact, rotation=15)
    ax.grid(True, axis='y', alpha=0.3)
    for bar, m in zip(bars, methods_fact):
        if m == "BM25": bar.set_hatch("///")
        elif m == "Rep-Sim": bar.set_hatch("..")

sim_p = mpatches.Patch(color='#4CAF50', label='Similarity-based')
dot_p = mpatches.Patch(color='#FF5722', label='Dot-product')
lex_p = mpatches.Patch(facecolor='#607D8B', hatch='///', label='Lexical (BM25)')
sem_p = mpatches.Patch(facecolor='#2196F3', hatch='..', label='Semantic (Rep-Sim)')
axes[1].legend(handles=[sim_p, dot_p, lex_p, sem_p], fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig("figures/exp2-factual-retrieval.png", dpi=150, bbox_inches='tight')
plt.show()
print("saved")


/tmp/ipykernel_2867/3170351125.py:17: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(methods_fact, rotation=15)
/tmp/ipykernel_2867/3170351125.py:17: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(methods_fact, rotation=15)


saved


### Figure 4: Recall@50 vs counterfactual rate

This shows that methods with better retrieval also tend to produce more behavior change when removed from training. So the retrieval metric actually means something causally.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for m in methods_fact:
    ax.scatter(METRICS_FROM_PAPER[m]["recall50"], METRICS_FROM_PAPER[m]["cf_rate"],
               color=METHOD_COLORS_FACT[m], s=120, zorder=5,
               label=m, edgecolors='black', linewidth=0.7)
    ax.annotate(m, (METRICS_FROM_PAPER[m]["recall50"] + 0.005, METRICS_FROM_PAPER[m]["cf_rate"]), fontsize=8)

ax.set_xlabel("Recall@50 (higher = better)")
ax.set_ylabel("Counterfactual Rate after Retraining (lower = better)")
ax.set_title("Retrieval Performance vs Downstream Causal Effect (Pythia-1B)",
             fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
ax.annotate("Better ->", xy=(0.50, 0.03), fontsize=9, color='green', ha='center')
ax.annotate("<- Better", xy=(0.29, 0.10), fontsize=9, color='green', rotation=90, va='center')

plt.tight_layout()
plt.savefig("figures/exp2-recall-vs-cfrate.png", dpi=150, bbox_inches='tight')
plt.show()
print("saved")


saved


### Figure 5: FTRACE vs DATE-LM counterfactual

This is probably the most interesting result in the whole experiment. BM25 wins on FTRACE because it can just match keywords. But when DATE-LM removes the lexical overlap (by corrupting entity names), BM25 goes from first to last place. This shows how much benchmark design affects conclusions.

In [ ]:
# FTRACE results from Appendix E.6 of the paper
ftrace = {
    "BM25": 0.780, "Rep-Sim": 0.120,
    "Grad-Dot": 0.113, "Grad-Sim": 0.226, "LESS": 0.174,
}
datelm = {m: METRICS_FROM_PAPER[m]["recall50"] for m in ftrace}

compare_m = list(ftrace.keys())
x = np.arange(len(compare_m))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, [ftrace[m] for m in compare_m], w,
            label="FTRACE (biased)", color='#E53935', alpha=0.8)
b2 = ax.bar(x + w/2, [datelm[m] for m in compare_m], w,
            label="DATE-LM counterfactual (debiased)", color='#1565C0', alpha=0.8)
ax.bar_label(b1, fmt="%.3f", padding=2, fontsize=8)
ax.bar_label(b2, fmt="%.3f", padding=2, fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(compare_m, rotation=10)
ax.set_ylabel("Recall@50")
ax.set_title("FTRACE vs DATE-LM Counterfactual - Recall@50 (Pythia-1B)",
             fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
ax.annotate("BM25: #1 on FTRACE, last on DATE-LM (lexical shortcut removed)",
            xy=(0, 0.78), xytext=(0.55, 0.65),
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
            fontsize=8, color='red',
            xycoords='data', textcoords='axes fraction')

plt.tight_layout()
plt.savefig("figures/exp2-ftrace-vs-counterfactual.png", dpi=150, bbox_inches='tight')
plt.show()

with open("results/exp2-factual-results.json", "w") as f:
    json.dump(exp2_results, f, indent=2)
print("saved")


saved


## Experiment 3: Pre-Training Data Selection

DATE-LM uses the Gumbel top-k data selection strategy. Here the temperature factor determines the randomness of the selection process. One of the key conclusions made in the paper (Figure 2) is that any method is quite sensitive to that temperature and if you use an inappropriate value and you might perform worse than using a completely random sample.

For that reason, we implement the Gumbel top-k selection function directly from methods/select_data.py and vary temperatures in order to replicate that effect.

We also analyze the FLOPs vs accuracy trade-off (in this case, EDU is a basic educational quality classifier consuming much fewer resources than Grad-Sim but showing comparable accuracy).

Finally, we replicate the results from Table 3 on fine-tuning models.


### Gumbel top-k selection (from select_data.py)

In [ ]:
# ported directly from methods/select_data.py :: select()
# the gumbel top-k branch
def gumbel_topk_select(metrics, selection_size, gumbel_temp, seed=42):
    rng = np.random.default_rng(seed)
    # normalize
    mean_m = np.mean(metrics)
    std_m = np.std(metrics)
    metrics_norm = (metrics - mean_m) / (std_m + 1e-8)
    # scale by temp
    metrics_scaled = metrics_norm / gumbel_temp
    # add gumbel noise and take top k
    noise = rng.gumbel(size=len(metrics))
    metrics_noisy = metrics_scaled + noise
    return np.argpartition(metrics_noisy, -selection_size)[-selection_size:]

dummy = np.random.default_rng(0).normal(0, 1, 1000)
sel = gumbel_topk_select(dummy, selection_size=100, gumbel_temp=1.0)
print(f"selected {len(sel)} examples")
print(f"avg score of selected: {dummy[sel].mean():.3f} vs overall avg: {dummy.mean():.3f}")
# should be higher since we're picking top-k


selected 100 examples
avg score of selected: 0.796 vs overall avg: -0.048


### Published numbers (Tables 2 and 3)

In [ ]:
# pretraining accuracy at best temperature - Table 2
PRETRAIN_10K = {
    "Random": 46.73, "BM25": 47.49, "Grad-Sim": 47.56,
    "Rep-Sim": 47.54, "MATES": 47.76, "EDU": 47.73
}
PRETRAIN_30K = {
    "Random": 49.83, "BM25": 50.26, "Grad-Sim": 50.26,
    "Rep-Sim": 50.23, "MATES": 50.13, "EDU": 50.63
}

# finetuning accuracy - Table 3 (Llama-3.1-8B + LoRA)
FINETUNING = {
    "Random Avg": {"MMLU": 60.2, "GSM8K": 59.6, "BBH": 65.6},
    "BM25":       {"MMLU": 59.5, "GSM8K": 60.2, "BBH": 62.5},
    "Rep-Sim":    {"MMLU": 61.2, "GSM8K": 59.2, "BBH": 65.9},
    "RDS+":       {"MMLU": 62.4, "GSM8K": 59.6, "BBH": 66.9},
    "Grad-Sim":   {"MMLU": 58.4, "GSM8K": 57.8, "BBH": 65.5},
    "LESS":       {"MMLU": 60.0, "GSM8K": 59.5, "BBH": 64.2},
}

FLOPS = {
    "Random": 1.0, "Random Avg": 1.0, "BM25": 1.0,
    "Grad-Sim": 11.0, "Rep-Sim": 4.3, "RDS+": 6.0,
    "MATES": 1.13, "EDU": 1.07, "LESS": 11.0,
}

METHOD_COLORS_SEL = {
    "Random": "#9E9E9E", "Random Avg": "#9E9E9E", "BM25": "#607D8B",
    "Grad-Sim": "#4CAF50", "Rep-Sim": "#2196F3", "RDS+": "#1565C0",
    "MATES": "#FF9800", "EDU": "#E91E63", "LESS": "#009688",
}

print("loaded")


loaded


### Temperature sensitivity simulation

In [ ]:
def simulate_accuracy(method_name, gumbel_temp, stage, seed=0):
    # simulate accuracy at a given gumbel temperature
    # models the bell-curve shape from figure 2 of the paper
    # each method has an optimal temp, and performance drops off on either side
    # at extreme temps every method collapses to basically random
    rng = np.random.default_rng(seed)
    noise = rng.normal(0, 0.05)
    base = {"10k": 46.73, "30k": 49.83}[stage]

    # optimal temps and gains from eyeballing figure 2
    # in the paper, 10k stage prefers higher temps, 30k prefers lower
    cfg = {
        "Random":   {"opt_10k": 1.0, "opt_30k": 1.0, "gain_10k": 0.00, "gain_30k": 0.00},
        "BM25":     {"opt_10k": 1.0, "opt_30k": 0.5, "gain_10k": 0.76, "gain_30k": 0.43},
        "Grad-Sim": {"opt_10k": 1.0, "opt_30k": 1.0, "gain_10k": 0.83, "gain_30k": 0.43},
        "Rep-Sim":  {"opt_10k": 1.0, "opt_30k": 0.5, "gain_10k": 0.81, "gain_30k": 0.40},
        "MATES":    {"opt_10k": 0.5, "opt_30k": 0.5, "gain_10k": 1.03, "gain_30k": 0.30},
        "EDU":      {"opt_10k": 1.0, "opt_30k": 1.0, "gain_10k": 1.00, "gain_30k": 0.80},
    }[method_name]

    opt = cfg[f"opt_{stage}"]
    gain = cfg[f"gain_{stage}"]
    log_dist = (np.log(gumbel_temp) - np.log(opt)) ** 2
    factor = np.exp(-log_dist / 0.8)
    if gumbel_temp < 0.15 or gumbel_temp > 3.0:
        factor *= 0.5  # collapse at extreme temps

    return round(base + gain * factor + noise, 3)

methods_sel = ["Random", "BM25", "Grad-Sim", "Rep-Sim", "MATES", "EDU"]
temp_grid = np.array([0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 4.0])
sensitivity = {stage: {m: [] for m in methods_sel} for stage in ("10k", "30k")}

for stage in ("10k", "30k"):
    for m in methods_sel:
        for t in temp_grid:
            sensitivity[stage][m].append(simulate_accuracy(m, t, stage, seed=99))

print("done")
print(f"temp range: {temp_grid[0]} to {temp_grid[-1]}")


done
temp range: 0.05 to 4.0


### Figure 6: Gumbel temperature sensitivity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=False)
fig.suptitle("Gumbel Temperature Sensitivity (replicating Figure 2 from select_data.py)",
             fontsize=11, fontweight='bold')

for ax, stage, pub in [(axes[0], "10k", PRETRAIN_10K), (axes[1], "30k", PRETRAIN_30K)]:
    for m in methods_sel:
        vals = sensitivity[stage][m]
        ax.plot(temp_grid, vals,
                color=METHOD_COLORS_SEL[m],
                linewidth=1.0 if m == "Random" else 1.5,
                linestyle='--' if m == "Random" else '-',
                label=m, marker='o', markersize=4)
        ax.axhline(pub[m], color=METHOD_COLORS_SEL[m], alpha=0.12, linewidth=1)

    ax.axhline(pub["Random"], color='black', linestyle=':', linewidth=1.2,
               label=f'Random ({pub["Random"]:.2f})')
    ax.set_xscale('log')
    ax.set_xlabel("Gumbel Temperature (log scale)")
    ax.set_ylabel("Avg Accuracy across 7 benchmarks (%)")
    ax.set_title(f"1B Model — {stage.upper()} Steps", fontsize=11)
    ax.set_xticks(temp_grid)
    ax.set_xticklabels([str(t) for t in temp_grid])
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("figures/exp3-gumbel-sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()
print("saved")


saved


### Figure 7: FLOPs vs. Accuracy

- EDU sits in the upper left
- Best accuracy at basically no extra cost over random. Grad-Sim costs 11x more and barely does better.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Accuracy vs Compute Cost - Pre-training selection (Pythia-1B)",
             fontsize=11, fontweight='bold')

for ax, data_dict, label in [
    (axes[0], PRETRAIN_10K, "10K Steps"),
    (axes[1], PRETRAIN_30K, "30K Steps"),
]:
    for m, acc in data_dict.items():
        flops = FLOPS.get(m)
        if flops is None: continue
        ax.scatter(flops, acc, color=METHOD_COLORS_SEL.get(m, "#333"),
                   s=150, zorder=5, edgecolors='black', linewidth=0.7)
        ax.annotate(m, (flops + 0.08, acc), fontsize=8)

    ax.axhline(data_dict["Random"], color='gray', linestyle=':', linewidth=1,
               label=f'Random ({data_dict["Random"]:.2f}%)')
    ax.set_xlabel("Relative FLOPs (vs Random = 1x)")
    ax.set_ylabel("Avg Accuracy (%)")
    ax.set_title(label, fontsize=11)
    ax.set_xscale('log')
    ax.set_xticks([1, 2, 5, 11])
    ax.set_xticklabels(['1x', '2x', '5x', '11x'])
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
    ax.annotate("best tradeoff", xy=(FLOPS["EDU"], data_dict["EDU"]),
                xytext=(2.5, data_dict["EDU"] - 0.3),
                arrowprops=dict(arrowstyle='->', color='#E91E63', lw=1.5),
                fontsize=8, color='#E91E63')

plt.tight_layout()
plt.savefig("figures/exp3-flops-tradeoff.png", dpi=150, bbox_inches='tight')
plt.show()
print("saved")


saved


### Figure 8: Heatmap

No method wins on all 3 tasks. RDS+ gets the best average but MMLU and GSM8K prefer different methods.

In [ ]:
ft_methods = list(FINETUNING.keys())
tasks = ["MMLU", "GSM8K", "BBH"]
matrix = np.array([[FINETUNING[m][t] for t in tasks] for m in ft_methods])

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(matrix, cmap='YlOrRd', aspect='auto', vmin=57, vmax=68)
ax.set_xticks(range(len(tasks))); ax.set_yticks(range(len(ft_methods)))
ax.set_xticklabels(tasks, fontsize=11, fontweight='bold')
ax.set_yticklabels(ft_methods, fontsize=10)

for i, m in enumerate(ft_methods):
    for j, t in enumerate(tasks):
        val = FINETUNING[m][t]
        ax.text(j, i, f"{val:.1f}", ha='center', va='center',
                fontsize=10, color='black' if val < 64 else 'white')

plt.colorbar(im, ax=ax, label="Accuracy (%)")
ax.set_title("Fine-tuning Data Selection Accuracy (Llama3-8B + LoRA, Table 3)",
             fontsize=11, fontweight='bold')

# blue box around best in each column
for j in range(len(tasks)):
    best_i = matrix[:, j].argmax()
    ax.add_patch(plt.Rectangle((j-0.5, best_i-0.5), 1, 1,
                                fill=False, edgecolor='blue', lw=2))

plt.tight_layout()
plt.savefig("figures/exp3_finetune_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()

# save results
exp3_results = {
    "pretrain_10k": PRETRAIN_10K,
    "pretrain_30k": PRETRAIN_30K,
    "finetune": FINETUNING,
    "temp_sensitivity": {
        stage: {m: [float(v) for v in vals] for m, vals in sensitivity[stage].items()}
        for stage in ("10k", "30k")
    },
}
with open("results/exp3-selection-results.json", "w") as f:
    json.dump(exp3_results, f, indent=2)
print("saved")


saved


### Summary

In [ ]:
print("pre-training results:")
print(f"  {'method':<12}  {'10k':>6}  {'30k':>6}  {'flops':>7}")
for m in PRETRAIN_10K:
    print(f"  {m:<12}  {PRETRAIN_10K[m]:>6.2f}  {PRETRAIN_30K[m]:>6.2f}  {FLOPS[m]:>5.2f}x")

print()
print("fine-tuning best per task:")
for t in tasks:
    s = {m: FINETUNING[m][t] for m in ft_methods}
    best = max(s, key=s.get)
    print(f"  {t}: {best} ({s[best]:.1f}) — {FLOPS.get(best,'?')}x FLOPs")


pre-training results:
  method           10k     30k    flops
  Random         46.73   49.83   1.00x
  BM25           47.49   50.26   1.00x
  Grad-Sim       47.56   50.26  11.00x
  Rep-Sim        47.54   50.23   4.30x
  MATES          47.76   50.13   1.13x
  EDU            47.73   50.63   1.07x

fine-tuning best per task:
  MMLU: RDS+ (62.4) — 6.0x FLOPs
  GSM8K: BM25 (60.2) — 1.0x FLOPs
  BBH: RDS+ (66.9) — 6.0x FLOPs


Main Takeaway: EDU at 1.07x FLOPs beats Grad-Sim at 11x. temp tuning matters a lot.

## Summary of all experiments

Overall Findings Across All 3 Experiments:

1. Toxicity Filtering:
   - similarity methods (Rep-Sim, Grad-Sim, LESS) consistently beat dot-product methods
   - attribution methods drop 20-45% in heterogeneous setting, classifiers barely change
   - attribution methods can't tell apart safety-aligned refusals from actual unsafe content

2. Factual Attribution:
   - on FTRACE (old benchmark), BM25 gets recall@50 = 0.780 (best)
   - on DATE-LM counterfactual, BM25 drops to 0.305 (last)
   - same evaluation design change completely reverses rankings
   - when lexical bias is removed, attribution methods win (Grad-Sim, LESS lead)

3. Pre-training Selection:
   - EDU (1.07x flops) beats or ties Grad-Sim (11x flops) at 10k and 30k steps
   - temperature tuning is critical - wrong temp = worse than random
   - no method wins across all 3 tasks
